In [5]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RESULTS_ROOT = Path("/mnt/primary/Baseline/baseline_results")

RESULT_FILES = {
    "Linear Regression": RESULTS_ROOT / "LinearRegression_company_rmse.csv",
    "Elastic Net": RESULTS_ROOT / "ElasticNet_company_rmse.csv",
    "SVR": RESULTS_ROOT / "SVR_company_rmse.csv",
    "Random Forest": RESULTS_ROOT / "RandomForest_company_rmse.csv",
    "XGBoost": RESULTS_ROOT / "XGBoost_company_rmse.csv",
}

for model_name, path in RESULT_FILES.items():
    print(model_name, "->", path, "| exists:", path.exists())


Linear Regression -> /mnt/primary/Baseline/baseline_results/LinearRegression_company_rmse.csv | exists: True
Elastic Net -> /mnt/primary/Baseline/baseline_results/ElasticNet_company_rmse.csv | exists: True
SVR -> /mnt/primary/Baseline/baseline_results/SVR_company_rmse.csv | exists: True
Random Forest -> /mnt/primary/Baseline/baseline_results/RandomForest_company_rmse.csv | exists: True
XGBoost -> /mnt/primary/Baseline/baseline_results/XGBoost_company_rmse.csv | exists: True


In [6]:
frames = []

for model_name, path in RESULT_FILES.items():
    if not path.exists():
        print("Missing results file:", path)
        continue

    frame = pd.read_csv(path)
    frames.append(frame)

if not frames:
    raise FileNotFoundError("No baseline RMSE result files were found.")

company_results = pd.concat(frames, ignore_index=True)


In [7]:
final_results = (
    company_results.groupby(["ML_Model", "FeatureSet", "Horizon_Days"], as_index=False)[["Validation_RMSE", "Test_RMSE"]]
    .mean()
    .sort_values(["Horizon_Days", "Test_RMSE", "ML_Model", "FeatureSet"])
    .reset_index(drop=True)
)

display(final_results.round(4))


,ML_Model,FeatureSet,Horizon_Days,Validation_RMSE,Test_RMSE
0,Elastic Net,Basic,1,3.0688,3.8800
1,Elastic Net,Advanced,1,3.0924,3.8925
2,Random Forest,Advanced,1,3.1197,3.9179
3,Linear Regression,Basic,1,3.1305,3.9550
4,Random Forest,Basic,1,3.2651,4.0161
5,XGBoost,Advanced,1,3.2761,4.0371
6,Linear Regression,Advanced,1,3.3602,4.1195
7,XGBoost,Basic,1,3.5701,4.2819
8,SVR,Advanced,1,4.1042,5.0052
9,SVR,Basic,1,10.1403,10.4367


In [8]:
wide_test = (
    final_results.pivot_table(index=["ML_Model", "FeatureSet"], columns="Horizon_Days", values="Test_RMSE")
    .rename(columns={1: "Test RMSE 1D", 3: "Test RMSE 3D", 5: "Test RMSE 5D"})
    .reset_index()
)

wide_test.columns.name = None
display(wide_test.round(4))

wide_test.to_csv(RESULTS_ROOT / "baseline_test_rmse_comparison.csv", index=False)
final_results.to_csv(RESULTS_ROOT / "baseline_validation_test_rmse.csv", index=False)

print("Saved comparison results to:", RESULTS_ROOT.resolve())


,ML_Model,FeatureSet,Test RMSE 1D,Test RMSE 3D,Test RMSE 5D
0,Elastic Net,Advanced,3.8925,6.6705,8.6884
1,Elastic Net,Basic,3.8800,6.6037,8.5238
2,Linear Regression,Advanced,4.1195,7.4121,9.9044
3,Linear Regression,Basic,3.9550,6.9129,9.0730
4,Random Forest,Advanced,3.9179,6.7643,8.7491
5,Random Forest,Basic,4.0161,6.9975,8.9644
6,SVR,Advanced,5.0052,9.2439,12.2145
7,SVR,Basic,10.4367,18.4540,22.0792
8,XGBoost,Advanced,4.0371,7.0232,9.1587
9,XGBoost,Basic,4.2819,7.3968,9.3937


Saved comparison results to: /mnt/primary/Baseline/baseline_results
